<a href="https://colab.research.google.com/github/thecodemaster152/rice-type-classification/blob/main/Rice_type_classification_using_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install opendatasets --quiet
import opendatasets as od
od.download("https://www.kaggle.com/datasets/mssmartypants/rice-type-classification")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: thefinman152
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/mssmartypants/rice-type-classification


100%|██████████| 888k/888k [00:00<00:00, 124MB/s]

In [2]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [3]:
data_df = pd.read_csv("/content/rice-type-classification/riceClassification.csv")

In [4]:
data_df.head()

,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,1,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,4,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,5,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [5]:
data_df.describe()

,id,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
count,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000,18185.000000
mean,9093.000000,7036.492989,151.680754,59.807851,0.915406,7225.817872,94.132952,0.616653,351.606949,0.707998,2.599081,0.549079
std,5249.701658,1467.197150,12.376402,10.061653,0.030575,1502.006571,9.906250,0.104389,29.500620,0.067310,0.434836,0.497599
min,1.000000,2522.000000,74.133114,34.409894,0.676647,2579.000000,56.666658,0.383239,197.015000,0.174590,1.358128,0.000000
25%,4547.000000,5962.000000,145.675910,51.393151,0.891617,6125.000000,87.126656,0.538530,333.990000,0.650962,2.208527,0.000000
50%,9093.000000,6660.000000,153.883750,55.724288,0.923259,6843.000000,92.085696,0.601194,353.088000,0.701941,2.602966,1.000000
75%,13639.000000,8423.000000,160.056214,70.156593,0.941372,8645.000000,103.559146,0.695664,373.003000,0.769280,2.964101,1.000000
max,18185.000000,10210.000000,183.211434,82.550762,0.966774,11008.000000,114.016559,0.886573,508.511000,0.904748,3.911845,1.000000


In [6]:
data_df.dropna(inplace= True)
data_df = data_df.iloc[:,1:]

In [7]:
data_df.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,4537,92.229316,64.012769,0.719916,4677,76.004525,0.657536,273.085,0.764510,1.440796,1
1,2872,74.691881,51.400454,0.725553,3015,60.471018,0.713009,208.317,0.831658,1.453137,1
2,3048,76.293164,52.043491,0.731211,3132,62.296341,0.759153,210.012,0.868434,1.465950,1
3,3073,77.033628,51.928487,0.738639,3157,62.551300,0.783529,210.657,0.870203,1.483456,1
4,3693,85.124785,56.374021,0.749282,3802,68.571668,0.769375,230.332,0.874743,1.510000,1


In [8]:
from itertools import groupby
data_df['Class'].value_counts()

,count
Class,
1,9985
0,8200


In [9]:
original_df = data_df.copy()
for column in data_df.columns:
  data_df[column] = data_df[column]/data_df[column].abs().max()

data_df.head()

,Area,MajorAxisLength,MinorAxisLength,Eccentricity,ConvexArea,EquivDiameter,Extent,Perimeter,Roundness,AspectRation,Class
0,0.444368,0.503404,0.775435,0.744658,0.424873,0.666610,0.741661,0.537029,0.844997,0.368316,1.0
1,0.281293,0.407681,0.622653,0.750489,0.273892,0.530370,0.804230,0.409661,0.919215,0.371471,1.0
2,0.298531,0.416421,0.630442,0.756341,0.284520,0.546380,0.856278,0.412994,0.959862,0.374747,1.0
3,0.300979,0.420463,0.629049,0.764024,0.286791,0.548616,0.883772,0.414262,0.961818,0.379222,1.0
4,0.361704,0.464626,0.682901,0.775033,0.345385,0.601418,0.867808,0.452954,0.966836,0.386007,1.0


In [10]:
X = np.array(data_df.iloc[:, :-1])
y = np.array(data_df.iloc[:, -1])

In [11]:
X,y

(array([[0.44436827, 0.50340371, 0.77543522, ..., 0.5370287 , 0.844997  ,
         0.36831616],
        [0.28129285, 0.40768133, 0.62265269, ..., 0.40966075, 0.91921498,
         0.37147093],
        [0.29853085, 0.41642141, 0.63044229, ..., 0.41299402, 0.95986205,
         0.37474651],
        ...,
        [0.62340842, 0.84480035, 0.64091576, ..., 0.67304935, 0.75472018,
         0.74783024],
        [0.58374143, 0.8263563 , 0.62355087, ..., 0.67524793, 0.70210346,
         0.75187447],
        [0.60078355, 0.83554818, 0.62495614, ..., 0.6658912 , 0.74305096,
         0.7585284 ]]),
 array([1., 1., 1., ..., 0., 0., 0.]))

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3 )

In [13]:
X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size = 0.5 )

In [14]:
X_train.shape, X_test.shape, X_val.shape

((12729, 10), (2728, 10), (2728, 10))

In [15]:
class dataset(Dataset):
  def __init__(self,X,y):
    self.X = torch.tensor(X , dtype =torch.float32).to(device)
    self.Y = torch.tensor(y , dtype =torch.float32).to(device)

  def __len__(self):
      return len(self.X)

  def __getitem__(self, idx):
      return self.X[idx], self.Y[idx]

In [16]:
trainig_data = dataset(X_train , y_train)
testing_data = dataset(X_test , y_test)
validation_data = dataset(X_val , y_val)

In [17]:
train_dataloader = DataLoader(trainig_data, batch_size=8, shuffle=True)
test_dataloader = DataLoader(testing_data, batch_size=8, shuffle=True)
val_dataloader = DataLoader(validation_data, batch_size=8, shuffle=True)

In [18]:
train_dataloader

In [19]:
for X,y in train_dataloader:
  print(X)
  print("++++++++++++++++++++++++++")
  print(y)
  break


tensor([[0.7961, 0.8880, 0.7758, 0.9509, 0.7515, 0.8922, 0.8356, 0.7341, 0.8101,
         0.6494],
        [0.8907, 0.8975, 0.8580, 0.9335, 0.8403, 0.9438, 0.7388, 0.7530, 0.8614,
         0.5935],
        [0.8340, 0.8450, 0.8608, 0.9190, 0.7924, 0.9132, 0.7211, 0.7282, 0.8625,
         0.5569],
        [0.8362, 0.8622, 0.8434, 0.9285, 0.7951, 0.9145, 0.7214, 0.7370, 0.8444,
         0.5800],
        [0.5468, 0.7573, 0.6272, 0.9596, 0.5243, 0.7395, 0.5823, 0.6219, 0.7753,
         0.6850],
        [0.4803, 0.6828, 0.6127, 0.9461, 0.4567, 0.6930, 0.6836, 0.5728, 0.8028,
         0.6323],
        [0.6217, 0.7457, 0.7248, 0.9299, 0.5924, 0.7885, 0.6594, 0.6353, 0.8449,
         0.5837],
        [0.9170, 0.8815, 0.9009, 0.9182, 0.8733, 0.9576, 0.7065, 0.7911, 0.8035,
         0.5551]], device='cuda:0')
++++++++++++++++++++++++++
tensor([0., 0., 0., 0., 1., 1., 0., 0.], device='cuda:0')


In [25]:
HIDDEN_NEURONS = 10
class MyModel(nn.Module):
  def __init__(self):
    super(MyModel,self).__init__()

    self.input_layer = nn.Linear(X.shape[1], HIDDEN_NEURONS)

    self.linear = nn.Linear(HIDDEN_NEURONS, 1)

    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    x = self.input_layer(x)
    x = self.linear(x)
    x = self.sigmoid(x)
    return x


model = MyModel().to(device)